# Explore Saved Evaluation Results

Use pandas and Matplotlib to explore outcomes, every metric topic, agent roles,
who affected whom, and selected replays. MARL-BGs owns file reading, metric
meanings and uncertainty. No cell trains, plays games, downloads models or
opens a Viewer. Install the optional `viz` and `notebook` extras. On Colab or
Kaggle, the first code cell installs them for you:
[Open in Colab](https://colab.research.google.com/github/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/examples/evaluation_analysis.ipynb) · [Open in Kaggle](https://kaggle.com/kernels/welcome?src=https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/examples/evaluation_analysis.ipynb).

Choose `MARL_BGS_RESULTS` before starting the kernel, or use the packaged
software demonstration: two two-tick games with custom counter Systems whose
actions use the Random policy, all ten agents active, one full record and two
replays. These tiny games show the tools, not learned skill. They cannot support
reliable scientific conclusions. Missing uncertainty remains unavailable.

Game seeds are not independent training seeds. A behavior count alone does not
prove that the behavior helped. The detailed charts below describe one saved
game; they are not method rankings or pooled estimates.

In [ ]:
# On Colab or Kaggle, install MARL-BattleGrounds with the JAX build that
# matches this notebook's accelerator. Anywhere else this cell does nothing;
# install first with https://marl-battlegrounds.com/#start.
import importlib.metadata
import os
import subprocess
import sys

if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    installed = {
        (distribution.metadata["Name"] or "").lower()
        for distribution in importlib.metadata.distributions()
    }
    builds = (("jax-cuda13-plugin", "cuda13"), ("jax-cuda12-plugin", "cuda12"))
    builds += (("libtpu", "tpu"),)
    build = next((name for plugin, name in builds if plugin in installed), None)
    extras = "all" if build is None else f"all,{build}"
    package = f"marl-battlegrounds[{extras}]==1.0.0"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    print(f"Installed {package}.")

## Choose Input And Output

Set `MARL_BGS_RESULTS` to the exact folder containing `run_details.json`.
Set `MARL_BGS_REPORT_DIR` to a new or empty folder for tables and figures.
For a fresh complete execution, choose a fresh destination. Existing reports are
preserved; results from two executions must not silently share an output folder.
`MARL_BGS_PHASE` and `MARL_BGS_PASS_ID` optionally select a saved pass.
A wrong supplied path is an error; it never silently falls back to the demo.

In [ ]:
import json
import math
import os
import shlex
import tempfile
import textwrap
from collections.abc import Mapping
from contextlib import ExitStack
from importlib.resources import as_file, files
from pathlib import Path
from typing import Protocol

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import JSON, Image, display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from numpy.typing import NDArray

import marl_battlegrounds as marl_bgs

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
    }
)
TEAM_COLORS = {"Team A": "#2166ac", "Team B": "#b35806"}

resources = ExitStack()
requested = os.environ.get("MARL_BGS_RESULTS")
using_demo = requested is None
result_dir = (
    Path(
        resources.enter_context(
            as_file(files("marl_battlegrounds").joinpath("examples_data", "evaluation"))
        )
    )
    if using_demo
    else Path(requested).expanduser()
).resolve()
if not (result_dir / "run_details.json").is_file():
    raise FileNotFoundError(
        "Choose the exact saved result folder containing run_details.json: "
        f"{result_dir}"
    )
chosen_report = os.environ.get("MARL_BGS_REPORT_DIR")
report_dir = (
    Path(chosen_report).expanduser()
    if chosen_report
    else Path(tempfile.mkdtemp(prefix="marl-bgs-evaluation-report-"))
).resolve()
if report_dir == result_dir or result_dir in report_dir.parents:
    raise ValueError("Keep report files outside the saved input folder")
if report_dir.exists() and (not report_dir.is_dir() or any(report_dir.iterdir())):
    raise FileExistsError(
        "Choose a new or empty report folder; existing files are preserved: "
        f"{report_dir}"
    )
report_dir.mkdir(parents=True, exist_ok=True)
phase = os.environ.get("MARL_BGS_PHASE")
pass_id = os.environ.get("MARL_BGS_PASS_ID")
print("Input:", result_dir)
print("Report:", report_dir)
print("Kernel directory:", Path.cwd())
print("Package:", marl_bgs.__file__)
print("Software demonstration:", using_demo)


def clean(value: object) -> object:
    """Return JSON-safe saved values; preserve missing numbers as None, not zero."""
    if isinstance(value, np.ndarray):
        return clean(value.tolist())
    if isinstance(value, np.generic):
        return clean(value.item())
    if isinstance(value, dict):
        return {str(key): clean(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [clean(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def rows(columns: Mapping[str, NDArray[np.generic]], limit: int = 20) -> list[object]:
    """Return at most limit display rows from the public column mapping."""
    length = min(len(next(iter(columns.values()), ())), limit)
    return [
        clean({name: values[index] for name, values in columns.items()})
        for index in range(length)
    ]


report = {
    "input": str(result_dir),
    "software_demo": using_demo,
    "kernel_directory": str(Path.cwd()),
    "package": marl_bgs.__file__,
}


def save_figure(fig: Figure, name: str, data: pd.DataFrame | None = None) -> None:
    """Save and show a figure, with its optional pandas data and source identity.

    fig is a Matplotlib figure; name is a filename stem inside report_dir.
    data is an optional DataFrame containing exactly the plotted records.
    PNG and SVG files are written, optional CSV data is saved, and the figure
    is closed after display. The chart index retains the selected game or scope.
    """
    fig.savefig(report_dir / f"{name}.png", dpi=160, bbox_inches="tight")
    fig.savefig(report_dir / f"{name}.svg", bbox_inches="tight")
    if data is not None:
        data.to_csv(report_dir / f"{name}.csv", index=False)
    report.setdefault("charts", []).append(
        {
            "name": name,
            "png": f"{name}.png",
            "svg": f"{name}.svg",
            "data": f"{name}.csv" if data is not None else None,
            "selected_game": report.get("detail_game"),
            "saved_scope": clean(result.metadata["scope"]),
        }
    )
    display(Image(filename=str(report_dir / f"{name}.png")))
    plt.close(fig)


def table_rows(frame: pd.DataFrame) -> list[dict[str, object]]:
    """Return JSON-safe table records; pandas keeps missing numbers as null."""
    return json.loads(frame.to_json(orient="records"))

## Load One Saved Scope

The reader checks the stored files without repairing or rewriting them. It keeps
actual System identities, rosters, map conditions and pass ownership. It does not
choose the newest pass. Reload after a later resume: this object is a snapshot.

In [ ]:
result = marl_bgs.load_results(result_dir, phase=phase, pass_id=pass_id)
identity = {
    key: result.metadata.get(key)
    for key in (
        "run_id",
        "scope",
        "metric_schema_version",
        "status",
        "systems",
        "configurations",
    )
}
report["identity"] = clean(identity)
display(JSON(report["identity"]))

## Check Coverage First

Complete, incomplete and failed runs are different. Check scheduled and completed
games, spawn pairing and table availability before interpreting a mean.
`disabled` means a measurement was not requested; `unavailable` means the saved
scope cannot supply it. Neither means measured zero. Equal game counts do not
imply equal numbers of environment transitions.

In [ ]:
coverage = {
    "pass_status": result.metadata["pass_status"],
    "spawn_balance": result.metadata["spawn_balance"],
    "tables": {
        name: {
            key: facts.get(key)
            for key in (
                "availability",
                "reason",
                "missing_historical_fields",
                "unavailable_columns",
                "missing_passes",
            )
        }
        for name, facts in result.metadata["tables"].items()
    },
}
report["coverage"] = clean(coverage)
display(JSON(report["coverage"]))

## Read Native Outcomes

The shared summary reports wins, draws, losses and native points. Team A stays
Team A when spawn ends exchange. Native points are not kill counts: a Red Zone
death can award two points. Head-to-head rows show both directed perspectives;
keep their System and opponent identities when comparing rows.

In [ ]:
headlines = {}
for scope in ("overall", "map", "spawn"):
    try:
        frame = pd.DataFrame(result.head_to_head(by=scope))
        headlines[scope] = table_rows(frame.head(20))
        print(scope.title(), "— first 20 rows")
        display(frame.head(20))
    except ValueError as error:
        if scope != "spawn" or not str(error).startswith(
            "Spawn head-to-head unavailable:"
        ):
            raise
        headlines[scope] = {"unavailable_reason": str(error)}
        print(str(error))
report["head_to_head"] = headlines

## Read Supported Uncertainty

The API owns the point-margin interval calculation. It keeps verified paired
spawn games together and labels every interval with `interval_status`:
"Available"; "Conditional: Sampling not verified" when a method's sampling is
unknown (the bounds are kept, and `conditional_interval_assumption` says what
they assume); or "Unavailable", with missing bounds and an `unavailable_reason`.
Check the status before reading a bar as supported. These are fixed-method game
intervals, not uncertainty across independent training runs. The default
bootstrap settings are 2,000 draws and seed 19044001. This notebook does not
fit a second interval.

In [ ]:
point_frame = pd.DataFrame(result.point_margin_summary())
report["point_margins"] = table_rows(point_frame.head(20))
display(point_frame.head(20))
finite = point_frame.dropna(subset=["mean_point_margin"]).head(20)
if not finite.empty:
    fig, ax = plt.subplots(figsize=(8, max(2.5, len(finite) * 0.5)))
    ax.barh(
        range(len(finite)), finite["mean_point_margin"], color="#2166ac", alpha=0.85
    )
    for position, row in enumerate(finite.itertuples()):
        if pd.notna(row.ci_low) and pd.notna(row.ci_high):
            ax.hlines(position, row.ci_low, row.ci_high, color="#222222", linewidth=2)
    labels = [
        f"{row.system_name} [{position + 1}]"
        for position, row in enumerate(finite.itertuples())
    ]
    ax.set_yticks(range(len(finite)), labels)
    ax.axvline(0, color="#777777", linewidth=0.7)
    ax.set_xlabel("Mean Point Margin (Native Points)")
    ax.set_title("Point Margins; Missing Intervals Stay Unavailable")
    fig.tight_layout()
    save_figure(fig, "point_margins", finite)
else:
    print("No completed point measurements are available for a plot.")

## Compare Systems From Their Own Side

`system_metrics` shows every System as Team A, whichever side it really
played, so `team_a_kills` is always that System's own kills and
`score_difference` its points minus its opponent's. Summaries combine games by
each column's catalog weight (fractions as total over total) and add 95%
intervals from whole spawn pairs, with the System's Elo when the run is a
tournament. This tiny demo is one counter System playing itself in two games,
with metrics for one game only, so its summary values are NaN and
`unavailable_reason` says the metrics cover only 1 of 2 played games; the
per-game view still shows the measured game. Real comparisons need every game
measured at the level you read, and pairs of games for intervals.

In [ ]:
system_frame = pd.DataFrame(result.system_metrics("priority_metrics"))
shown = [
    name
    for name in (
        "system_name",
        "games",
        "score_difference",
        "team_a_kills",
        "team_a_deaths",
        "interval_status",
        "unavailable_reason",
    )
    if name in system_frame
]
report["system_metrics"] = table_rows(system_frame[shown]) if shown else []
display(system_frame[shown] if shown else system_frame)
game_frame = pd.DataFrame(
    result.system_metrics(
        "full_metrics",
        by="game",
        columns=["team_a_kills", "team_b_kills", "score_difference"],
    )
)
display(game_frame)

## Inspect Recorded Game Seeds

This is a bounded preview of the first 20 recorded game rows, including their
run, pass, map/config and seed IDs. Both spawn legs can share a game seed. A new
pass or game seed is not a new independently trained model. Compare training
seeds through the checked study export in the final cell.

In [ ]:
iterator = result.iter_table("episodes", rows=20)
try:
    episode_preview = next(iterator, {})
finally:
    iterator.close()
episodes = pd.DataFrame(episode_preview)
seed_fields = (
    "run_id",
    "phase",
    "pass_id",
    "episode_id",
    "seed_id",
    "map_id",
    "config_id",
    "team_a_policy",
    "team_b_policy",
    "outcome",
    "episode_length",
    "team_a_score",
    "team_b_score",
)
game_preview = episodes.loc[:, [name for name in seed_fields if name in episodes]]
report["game_preview"] = table_rows(game_preview)
display(game_preview)
if not episodes.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    outcome_counts = episodes["outcome"].value_counts().reindex([1, 3, 2], fill_value=0)
    axes[0].bar(
        ["Team A Wins", "Draws", "Team B Wins"],
        outcome_counts,
        color=[TEAM_COLORS["Team A"], "#888888", TEAM_COLORS["Team B"]],
    )
    axes[0].set_ylabel("Recorded Games")
    axes[0].set_title("Outcomes In The First 20 Saved Games")
    axes[1].plot(
        range(len(episodes)),
        episodes["team_a_score"],
        "o-",
        color=TEAM_COLORS["Team A"],
        label="Team A",
    )
    axes[1].plot(
        range(len(episodes)),
        episodes["team_b_score"],
        "s--",
        color=TEAM_COLORS["Team B"],
        label="Team B",
    )
    axes[1].set(
        xlabel="Saved Row (Not Training Time)",
        ylabel="Native Points",
        title="Final Scores In This Preview",
    )
    axes[1].legend()
    fig.tight_layout()
    save_figure(fig, "game_outcomes", game_preview)

## Explore Every Metric Topic And Group

The catalog is the shared source of labels, topics, units, roles and cautions.
A tick is one environment step. Agent-ticks add agents' time; pair-ticks add
observed ally pairs' time. A fraction and a count are different units.
Some measurements belong in several topic views, for example damage from an
Ultimate ability. Those views reuse a saved value; they are not extra samples.

The next cell reads up to `MAX_METRIC_GAMES = 200` full rows, one at a time,
keeping a small set of team measurements across games. `DETAIL_ROW = 0` selects
one of those rows for the detailed charts and full catalog. Change these values
for your data. Inspect the printed run, pass and game identity. This is a bounded
preview, not necessarily a random sample. The reader parses each row's full CSV
width but retains the full wide record for only the selected game. The full table
already includes priority measurements; never append both tables and count twice.
Missing values stay missing. Inactive slots stay in the schema and are named.

In [ ]:
from marl_battlegrounds.evaluation.metric_catalog import (
    METRIC_COLUMNS_BY_NAME,
    METRIC_TOPICS,
    metric_guidance,
    metric_locations,
    metric_order_key,
    metric_topic_text,
)

DETAIL_ROW = 0
MAX_METRIC_GAMES = 200
# A bounded sample for inspection, not a random sample or a new selection rule.
BEHAVIOR_STEMS = (
    "damage_done",
    "healing_done",
    "effective_healing_done",
    "excess_healing",
    "rescues",
    "rescue_opportunities",
    "focus_fire_concentration",
    "focus_fire_steps",
    "multi_contributor_kills",
    "trap_breaks",
    "trap_mean_remaining_steps_at_break",
    "ally_distance_mean",
)
behavior_columns = [
    column
    for column in METRIC_COLUMNS_BY_NAME.values()
    if column.scope == "team" and column.stem in BEHAVIOR_STEMS
]
behavior_records = []
identity_names = (
    "run_id",
    "phase",
    "pass_id",
    "episode_id",
    "seed_id",
    "map_id",
    "config_id",
)
if type(MAX_METRIC_GAMES) is not int or MAX_METRIC_GAMES < 1:
    raise ValueError("MAX_METRIC_GAMES must be a positive integer")
if type(DETAIL_ROW) is not int or not 0 <= DETAIL_ROW < MAX_METRIC_GAMES:
    raise ValueError("DETAIL_ROW must select a row inside MAX_METRIC_GAMES")
examined_metric_games = 0
topics = pd.DataFrame(
    [
        {
            "topic": topic.name,
            "section": topic.section,
            "label": topic.label,
            "description": topic.description,
        }
        for topic in METRIC_TOPICS
    ]
)
display(topics)
full_info = result.metadata["tables"]["full_metrics"]
full_rows = []
if full_info["availability"] == "available":
    iterator = result.iter_table("full_metrics", rows=1)
    try:
        for saved_row in range(MAX_METRIC_GAMES):
            batch = next(iterator, None)
            if batch is None:
                break
            record = {name: values[0] for name, values in batch.items()}
            examined_metric_games += 1
            if saved_row == DETAIL_ROW:
                full_rows.append(record)
            for column in behavior_columns:
                if column.name in record:
                    behavior_records.append(
                        {
                            **{name: record.get(name) for name in identity_names},
                            "saved_row": saved_row,
                            "metric": column.stem,
                            "team": "Team A" if column.subjects[0] == 1 else "Team B",
                            "value": record[column.name],
                            "unit": column.unit,
                        }
                    )
    finally:
        iterator.close()
else:
    print("Full metrics:", full_info["availability"], full_info.get("reason"))

if examined_metric_games and not full_rows:
    raise ValueError(
        f"DETAIL_ROW has no saved full row; read {examined_metric_games} games"
    )
behavior = pd.DataFrame(
    behavior_records,
    columns=[*identity_names, "saved_row", "metric", "team", "value", "unit"],
)
behavior["value"] = pd.to_numeric(behavior["value"], errors="raise").replace(
    [np.inf, -np.inf], np.nan
)
behavior.to_csv(report_dir / "behavior_by_game.csv", index=False)
report["behavior_sample"] = {
    "limit": MAX_METRIC_GAMES,
    "examined_games": examined_metric_games,
    "selection": "First saved full rows; not necessarily random or representative",
}
print(
    "Full-metric games examined:",
    examined_metric_games,
    "(limit",
    MAX_METRIC_GAMES,
    ")",
)

metric_columns = [
    "name",
    "value",
    "label",
    "stem",
    "family",
    "scope",
    "unit",
    "subjects",
    "subject_role",
    "recipient_role",
    "description",
    "numerator",
    "denominator",
    "missing_when",
    "guidance",
]
metric_records = []
selected_game = None
class_ids = (0,) * 10
if full_rows:
    selected_game = full_rows[0]
    class_ids = tuple(
        int(selected_game[f"agent_{slot}_class_id"]) for slot in range(10)
    )
    for name, value in selected_game.items():
        column = METRIC_COLUMNS_BY_NAME.get(name)
        if column is None:
            continue
        record = {
            field: getattr(column, field)
            for field in metric_columns
            if field not in ("value", "guidance")
        }
        record.update(value=value, guidance=metric_guidance(column, class_ids)[1])
        metric_records.append(record)
metrics = pd.DataFrame(metric_records, columns=metric_columns)
metrics["value"] = pd.to_numeric(metrics["value"], errors="raise").replace(
    [np.inf, -np.inf], np.nan
)
report["detail_game"] = (
    clean(
        {
            name: selected_game.get(name)
            for name in (
                "run_id",
                "phase",
                "pass_id",
                "episode_id",
                "seed_id",
                "config_id",
                "map_id",
            )
        }
    )
    if selected_game
    else None
)
report["metric_preview"] = [
    clean(
        {name: row.get(name) for name in ("run_id", "phase", "pass_id", "episode_id")}
    )
    for row in full_rows
]
report["uncatalogued_full_columns"] = [
    name
    for name in full_info["columns"]
    if name not in METRIC_COLUMNS_BY_NAME and name not in episode_preview
]
display(JSON(report["detail_game"]))
display(metrics.head(12))
if not metrics.empty:
    metrics[["name", "value", "unit", "scope", "stem"]].to_csv(
        report_dir / "selected_metrics.csv", index=False
    )

locations = pd.DataFrame(
    [
        {
            "name": row.name,
            "topic": topic,
            "view": view,
            "scope": row.scope,
            "finite": pd.notna(row.value),
        }
        for row in metrics.itertuples(index=False)
        for topic, view in metric_locations(METRIC_COLUMNS_BY_NAME[row.name], class_ids)
    ],
    columns=["name", "topic", "view", "scope", "finite"],
)
scopes = ["episode", "team", "agent", "team_recipient", "source_recipient", "ally_pair"]
coverage_index = pd.MultiIndex.from_product(
    [topics["topic"], scopes], names=["topic", "scope"]
)
metric_coverage = (
    locations.groupby(["topic", "scope"])
    .agg(saved_columns=("name", "size"), finite_values=("finite", "sum"))
    .reindex(coverage_index, fill_value=0)
    .reset_index()
)
# Zero here means no available fields, never zero measured behavior.
metric_coverage["status"] = np.where(
    metric_coverage["saved_columns"] == 0,
    "No Saved Fields",
    np.where(
        metric_coverage["finite_values"] == 0,
        "No Finite Values",
        "Recorded Values Available",
    ),
)
metric_coverage.to_csv(report_dir / "topic_coverage.csv", index=False)
groups = (
    locations.groupby(["topic", "view", "scope"])
    .agg(saved_columns=("name", "size"), finite_values=("finite", "sum"))
    .reset_index()
)
groups.to_csv(report_dir / "metric_groups.csv", index=False)
report["topic_coverage"] = table_rows(metric_coverage)
report["metric_groups"] = table_rows(groups)
display(groups)
if not metrics.empty:
    matrix = (
        metric_coverage.pivot(index="topic", columns="scope", values="finite_values")
        .reindex(index=topics["topic"], columns=scopes)
        .astype(float)
    )
    fig, ax = plt.subplots(figsize=(10, 10))
    image = ax.imshow(np.log1p(matrix.to_numpy()), cmap="Blues", aspect="auto")
    for row in range(len(matrix)):
        for col in range(len(scopes)):
            ax.text(
                col,
                row,
                str(int(matrix.iloc[row, col])),
                ha="center",
                va="center",
                fontsize=8,
            )
    ax.set_xticks(range(len(scopes)), scopes, rotation=25, ha="right")
    ax.set_yticks(range(len(topics)), topics["label"])
    ax.set_title("Metric Coverage In The Selected Game (Finite Saved Fields)")
    fig.colorbar(image, ax=ax, label="Log(1 + Field Count); Numbers Show Exact Counts")
    fig.tight_layout()
    save_figure(fig, "metric_coverage", metric_coverage)

In [ ]:
class MetricRow(Protocol):
    """The scope and slot IDs read from one saved pandas table row."""

    scope: str
    subjects: tuple[int, ...]


CLASS_NAMES = {
    0: "Inactive",
    1: "Mage",
    2: "Warrior",
    3: "Hunter",
    4: "Rogue",
    5: "Priest",
}


def agent_label(slot: int) -> str:
    """Name a global slot using this saved game's class and fixed team ownership."""
    return f"{'A' if slot < 5 else 'B'}{slot % 5} {CLASS_NAMES[class_ids[slot]]}"


def subject_label(row: MetricRow) -> str:
    """Name team, agent and pair roles without changing their saved ownership."""
    if row.scope == "episode":
        return "Game"
    if row.scope == "team":
        return "Team A" if row.subjects[0] == 1 else "Team B"
    if row.scope == "team_recipient":
        return (
            f"Team {'A' if row.subjects[0] == 1 else 'B'} → "
            f"{agent_label(row.subjects[1])}"
        )
    separator = " ↔ " if row.scope == "ally_pair" else " → "
    return separator.join(agent_label(slot) for slot in row.subjects)


def metric_panel(ax: Axes, frame: pd.DataFrame, title: str) -> None:
    """Plot up to ten finite values of one metric, or label its missing evidence."""
    shown = frame.dropna(subset=["value"]).head(10)
    if shown.empty:
        ax.text(
            0.5,
            0.5,
            "No Finite Saved Measurement",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        labels = [subject_label(row) for row in shown.itertuples(index=False)]
        colors = [
            TEAM_COLORS["Team B"]
            if (
                row.subjects
                and (
                    row.subjects[0] == 2
                    if row.scope in ("team", "team_recipient")
                    else row.subjects[0] >= 5
                )
            )
            else TEAM_COLORS["Team A"]
            for row in shown.itertuples(index=False)
        ]
        ax.barh(range(len(shown)), shown["value"], color=colors, alpha=0.85)
        ax.set_yticks(range(len(shown)), labels)
        ax.invert_yaxis()
        ax.set_xlabel(
            f"{shown.iloc[0]['unit']} · {shown.iloc[0]['scope']}".replace(
                "_", " "
            ).title()
        )
        if (shown["value"] >= 0).all():
            ax.set_xlim(left=0)
        if (shown["value"] == 0).all():
            ax.set_xlim(0, 1)
        if (shown["value"] == 0).all():
            ax.text(
                0.98,
                0.5,
                "All Shown Values Are 0",
                ha="right",
                transform=ax.transAxes,
                color="#555555",
            )
    ax.set_title(textwrap.fill(title, 46), fontsize=10, loc="left")

## Question: Was Healing Useful, And Were Saves Possible?

Compare healing provided, useful healing and excess healing in the same health
units. Useful healing can offset damage during a tick even if health does not
rise or the ally still dies. Excess Basic healing can still accompany Freedom;
excess health alone does not prove a wasted action. The table keeps each metric's
catalog meaning and denominator.

Rescue opportunities give saves their context. The metric checks threatened
allies separately; the opportunities may not all be achievable at once. Zero saves with zero opportunities
is not failed rescue behavior. A missing rescue rate is not 0%. Even a recorded
save does not establish that the whole strategy was good. Trace the matching
source-recipient values and replay before making that claim.

In [ ]:
healing_stems = (
    "healing_done",
    "effective_healing_done",
    "excess_healing",
    "rescue_opportunities",
    "rescues",
    "rescue_rate",
)
healing = metrics.loc[
    (metrics["scope"] == "team") & metrics["stem"].isin(healing_stems)
].copy()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, stem in zip(axes.flat, healing_stems, strict=False):
    metric_panel(
        ax, healing.loc[healing["stem"] == stem], stem.replace("_", " ").title()
    )
fig.suptitle(
    "Healing And Rescue — Read The Opportunities Too", fontsize=14, fontweight="bold"
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
save_figure(fig, "healing_and_rescue", healing[["name", "value", "unit", "stem"]])
display(
    healing[["name", "value", "unit", "description", "denominator", "missing_when"]]
)
observations = []
for team_id, team_name in ((1, "Team A"), (2, "Team B")):
    values = {
        row.stem: row.value
        for row in healing.itertuples(index=False)
        if row.subjects == (team_id,)
    }
    if all(pd.notna(values.get(stem, np.nan)) for stem in healing_stems[:3]):
        observations.append(
            f"{team_name} recorded {values['healing_done']:g} health of healing, "
            f"{values['effective_healing_done']:g} useful and "
            f"{values['excess_healing']:g} excess. "
            "These are saved measurements in the selected game, not a policy ranking."
        )
    if values.get("rescue_opportunities") == 0:
        observations.append(
            f"{team_name} had no recorded rescue opportunities; "
            "this game cannot test rescue success."
        )
report["healing_observations"] = observations
for observation in observations:
    print(observation)

## Question: Did Joint Attacks And Control Create Useful Chances?

Read focus-fire concentration beside its observed multi-attacker ticks. A high fraction
based on very few ticks is thin evidence. It does not prove that the team chose
the right target. Multiple kill contributors show same-tick involvement under
the catalog rule; they do not prove that every contributor was necessary.

Trap breaks are not automatically mistakes. Breaking a trap may help finish a
kill, or may free an enemy too early. Remaining trap time tells us what control
was lost, not whether losing it was wrong. Use the replay to check timing, ally
positions and the information available when the action was chosen.

In [ ]:
coordination_stems = (
    "focus_fire_concentration",
    "focus_fire_steps",
    "multi_contributor_kills",
    "trap_intervals",
    "trap_breaks",
    "trap_mean_remaining_steps_at_break",
)
coordination = metrics.loc[
    (metrics["scope"] == "team") & metrics["stem"].isin(coordination_stems)
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, stem in zip(axes.flat, coordination_stems, strict=False):
    metric_panel(
        ax,
        coordination.loc[coordination["stem"] == stem],
        stem.replace("_", " ").title(),
    )
fig.suptitle(
    "Coordination And Control — Counts Need Context", fontsize=14, fontweight="bold"
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
save_figure(
    fig, "coordination_and_control", coordination[["name", "value", "unit", "stem"]]
)
display(
    coordination[
        ["name", "value", "unit", "description", "denominator", "missing_when"]
    ]
)

## Question: Does The Pattern Recur Across Recorded Games?

A single replay is a case study. The following plots show separate game values
for a bounded full-metric sample. Colors retain Team A and Team B. Each figure
keeps one exact config and pass, so different maps, rosters and passes are not
silently pooled. The CSV retains all read identities and values. For readability,
plot the first two recorded config/pass groups; this is not model selection.

The x-axis is saved row order, not learning time. Dots are games, not independent
training runs. Selected full-metric games may be biased; check how they were
selected before generalizing. This cell computes no new confidence interval and
does not average per-game ratios with different denominators.

In [ ]:
if behavior.empty:
    print("No full-metric games are available for behavior comparisons.")
else:
    groups = behavior.groupby(
        ["run_id", "phase", "pass_id", "config_id"], dropna=False, sort=False
    )
    print("Recorded config/pass groups:", groups.ngroups, "— plotting the first two")
    comparison_stems = (
        "effective_healing_done",
        "excess_healing",
        "focus_fire_concentration",
        "multi_contributor_kills",
        "trap_breaks",
        "ally_distance_mean",
    )
    for number, (identity, frame) in enumerate(groups):
        if number == 2:
            break
        fig, axes = plt.subplots(2, 3, figsize=(14, 7))
        for ax, stem in zip(axes.flat, comparison_stems, strict=False):
            chosen = frame.loc[frame["metric"] == stem].dropna(subset=["value"])
            for team_name, team in chosen.groupby("team", sort=False):
                ax.scatter(
                    team["saved_row"],
                    team["value"],
                    label=team_name,
                    color=TEAM_COLORS[team_name],
                    marker="o" if team_name == "Team A" else "x",
                )
            ax.set_title(stem.replace("_", " ").title(), fontsize=10, loc="left")
            ax.set_xlabel("Saved Row (Not Training Time)")
            if chosen.empty:
                ax.text(0.5, 0.5, "Unavailable", transform=ax.transAxes, ha="center")
            else:
                ax.set_ylabel(chosen.iloc[0]["unit"])
                ax.legend(fontsize=8)
        fig.suptitle(
            f"Behavior Across Games · Pass {identity[1]}/{identity[2]} · "
            f"Config {str(identity[3])[:10]}",
            fontsize=13,
            fontweight="bold",
        )
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        save_figure(fig, f"behavior_across_games_{number + 1}", frame)

## Compare Agents And Trace Recipients

These charts retain class, team and global-slot ownership. Each panel has one
metric and unit. Heatmaps use rows as sources and columns as recipients; read
the saved role labels. Gray cells are missing or not applicable, not zero.
The team-recipient panel starts from recorded team values, not sums of agent
values. Ally spacing keeps only recorded unordered pairs; its blank diagonal
and opposite triangle are not invented distances. Dead and inactive are different.

Damage can exceed remaining health. Healing provided is not automatically useful
healing. Shared kill contributions do not count unique kills. Team and agent
ratios have their own denominators: do not average them into a new success rate.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, stem in zip(
    axes.flat, ("damage_done", "healing_done", "deaths", "dead_steps"), strict=False
):
    selected = metrics.loc[(metrics["scope"] == "agent") & (metrics["stem"] == stem)]
    metric_panel(ax, selected, "Agent " + stem.replace("_", " ").title())
fig.suptitle(
    "Agent Contributions And Costs — One Saved Game", fontsize=14, fontweight="bold"
)
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_figure(
    fig,
    "agent_contributions",
    metrics.loc[
        (metrics["scope"] == "agent")
        & metrics["stem"].isin(("damage_done", "healing_done", "deaths", "dead_steps")),
        ["name", "value", "unit", "scope", "stem"],
    ],
)


def pair_panel(ax: Axes, scope: str, stem: str, title: str) -> pd.DataFrame:
    """Show saved directed or unordered pairs without filling missing values."""
    selected = metrics.loc[
        (metrics["scope"] == scope) & (metrics["stem"] == stem)
    ].copy()
    if selected.empty or selected["value"].notna().sum() == 0:
        metric_panel(ax, selected, title)
        return selected
    selected["source"] = selected["subjects"].map(lambda value: value[0])
    selected["recipient"] = selected["subjects"].map(lambda value: value[1])
    matrix = selected.pivot(index="source", columns="recipient", values="value")
    sources = [1, 2] if scope == "team_recipient" else list(range(10))
    matrix = matrix.reindex(index=sources, columns=range(10))
    cmap = plt.get_cmap("YlGnBu").copy()
    cmap.set_bad("#eeeeee")
    values = np.ma.masked_invalid(matrix.to_numpy(dtype=float))
    image = ax.imshow(
        values, cmap=cmap, aspect="auto", vmin=0, vmax=max(1.0, float(values.max()))
    )
    if float(values.max()) == 0:
        title += " (All Recorded Values Are 0)"
    ax.set_xticks(
        range(10),
        [agent_label(slot) for slot in range(10)],
        rotation=60,
        ha="right",
        fontsize=8,
    )
    ax.set_yticks(
        range(len(sources)),
        ["Team A", "Team B"]
        if scope == "team_recipient"
        else [agent_label(slot) for slot in sources],
        fontsize=8,
    )
    ax.set_xlabel("Recipient" if scope != "ally_pair" else "Second Ally")
    ax.set_ylabel("Source" if scope != "ally_pair" else "First Ally")
    ax.set_title(title, loc="left", fontsize=11)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            if not np.ma.is_masked(values[row, col]) and values[row, col] != 0:
                ax.text(
                    col,
                    row,
                    f"{values[row, col]:.3g}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white"
                    if values[row, col] > float(values.max()) * 0.6
                    else "black",
                )
    ax.figure.colorbar(
        image, ax=ax, fraction=0.04, pad=0.03, label=selected.iloc[0]["unit"]
    )
    return selected


fig, axes = plt.subplots(2, 2, figsize=(14, 12))
relationship_frames = []
for ax, item in zip(
    axes.flat,
    (
        ("source_recipient", "damage_done", "Who Damaged Whom"),
        ("source_recipient", "healing_done", "Who Healed Whom (All Healing Provided)"),
        (
            "team_recipient",
            "basic_applications",
            "Team Basic Ability Uses By Recipient",
        ),
        ("ally_pair", "ally_distance_mean", "Mean Ally Spacing (Recorded Pairs Only)"),
    ),
    strict=False,
):
    relationship_frames.append(pair_panel(ax, *item))
fig.suptitle(
    "Sources, Recipients And Ally Pairs — One Saved Game",
    fontsize=14,
    fontweight="bold",
)
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_figure(
    fig,
    "relationships",
    pd.concat(relationship_frames, ignore_index=True)[
        ["name", "value", "unit", "scope", "stem", "subjects"]
    ],
)

## Read The Topic Gallery

Every topic gets a panel. Each panel shows **one named metric**, in one unit and
one scope, for the selected game. The choice follows stable catalog order:
prefer team totals, then agent values, then other scopes. It does not choose the
largest value or combine unlike measurements. The CSVs list all topic/group
coverage, and the `metrics` DataFrame keeps every catalogued saved measurement.

Zero-length bars mean recorded zero. Missing values have no bar. An empty panel
explains that no finite measurement is available. More is not always better;
read the description and guidance table before interpreting any panel.
Plots describe this one game, not uncertainty or overall method quality.

In [ ]:
gallery = []
for section, section_topics in topics.groupby("section", sort=False):
    plotted = []
    fig, axes = plt.subplots(
        (len(section_topics) + 1) // 2,
        2,
        figsize=(13, 3.8 * ((len(section_topics) + 1) // 2)),
        squeeze=False,
    )
    for ax, topic in zip(
        axes.flat, section_topics.itertuples(index=False), strict=False
    ):
        names = locations.loc[locations["topic"] == topic.topic, "name"]
        available = metrics.loc[metrics["name"].isin(names)].dropna(subset=["value"])
        choice = None
        for scope in (
            "team",
            "agent",
            "episode",
            "team_recipient",
            "source_recipient",
            "ally_pair",
        ):
            candidates = available.loc[available["scope"] == scope]
            if not candidates.empty:
                name = min(
                    candidates["name"],
                    key=lambda name: metric_order_key(
                        METRIC_COLUMNS_BY_NAME[name], topic.topic
                    ),
                )
                choice = METRIC_COLUMNS_BY_NAME[name]
                break
        selected = metrics.iloc[:0]
        if choice is not None:
            selected = available.loc[
                (available["stem"] == choice.stem)
                & (available["scope"] == choice.scope)
                & (available["unit"] == choice.unit)
            ]
            text = metric_topic_text(choice, topic.topic, class_ids)
            description = {
                field: text.get(field, getattr(choice, field))
                for field in (
                    "label",
                    "description",
                    "numerator",
                    "denominator",
                    "missing_when",
                )
            }
            description.update(
                topic=topic.topic,
                name=choice.name,
                unit=choice.unit,
                guidance=text.get("guidance", metric_guidance(choice, class_ids)[1]),
            )
            gallery.append(description)
        label = topic.label + ("\n" + choice.stem if choice else "")
        metric_panel(ax, selected, label)
        plotted.append(
            selected.dropna(subset=["value"]).head(10).assign(topic=topic.topic)
        )
    for ax in list(axes.flat)[len(section_topics) :]:
        ax.set_visible(False)
    fig.suptitle(section + " — One Saved Game", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    save_figure(
        fig,
        "topics_" + section.lower().replace(",", "").replace(" ", "_"),
        pd.concat(plotted, ignore_index=True)[
            ["topic", "name", "value", "unit", "scope", "stem"]
        ],
    )
report["metric_meanings"] = clean(gallery)
display(pd.DataFrame(gallery))

## Inspect The Matching Replay

The table joins the recorded pass/game mapping to verified replay paths, without
parsing filenames. It marks the selected detail game. No matching replay means
unavailable; the notebook never substitutes a more impressive game. It verifies
that one replay and prints the Viewer command, but starts no Viewer.

Watch the relevant ticks: who was threatened, who acted, what each actor could
know, and whether the support or control coincided with a useful outcome. Check
failed cases too. This can suggest an explanation; a controlled comparison is
needed to establish a cause. Dead and padded slots keep their saved identities.


In [ ]:
from marl_battlegrounds.evaluation.replay_io import load_replay

replay_paths = result.replay_paths
verified_paths = {path.resolve() for path in replay_paths}
report["replays"] = [str(path) for path in replay_paths]
replay_rows = []
for entry in result.metadata["passes"].values():
    for episode_id, saved in entry.get("replays", {}).items():
        path = (result_dir / saved["path"]).resolve()
        if path not in verified_paths:
            continue
        same_game = selected_game is not None and all(
            str(left) == str(right)
            for left, right in (
                (entry["phase"], selected_game["phase"]),
                (entry["pass_id"], selected_game["pass_id"]),
                (episode_id, selected_game["episode_id"]),
            )
        )
        replay_rows.append(
            {
                "phase": entry["phase"],
                "pass_id": entry["pass_id"],
                "episode_id": episode_id,
                "config_id": saved.get("config_id"),
                "saved_outcome_length_scores": saved.get("summary"),
                "selected_detail_game": same_game,
                "path": str(path),
            }
        )
replay_table = pd.DataFrame(replay_rows)
report["replay_index"] = replay_rows
display(replay_table.head(20))
selected_replay = next(
    (row for row in replay_rows if row["selected_detail_game"]), None
)
if selected_replay:
    replay = load_replay(selected_replay["path"]).replay
    print("Verified selected game replay:", type(replay).__name__)
    print(
        "python -m marl_battlegrounds replay",
        shlex.quote(selected_replay["path"]),
        "--no-open",
    )
else:
    print(
        "No replay exists for the selected detail game; no other game is substituted."
    )
if not replay_table.empty:
    replay_table.to_csv(report_dir / "replay_index.csv", index=False)

## Turn Observations Into A Testable Claim

Write down the claim before comparing final test results. For example: “Method X
uses healing to keep exposed allies alive more often than Method Y.” Then:

1. Compare held-out outcomes under the same declared maps, rosters and opponent
   conditions. Use independently trained seeds for claims about a method.
2. Check opportunity counts as well as saves, useful healing, deaths and outcomes.
   More healing can also mean taking more avoidable damage.
3. Inspect matched replays, including failures. Preserve game IDs; do not report
   only the most impressive clip. Separate a hypothesis from a confirmed result.
4. Use a controlled ablation to test the proposed cause. A correlation between
   focus fire and wins does not show which caused which.

A useful results paragraph gives the comparison, the number and kind of
independent units, the measured difference with supported uncertainty, the
behavioral evidence, and its limits. The two-game demo is a worked inspection,
not material for such a scientific claim. For a real study, use the checked
study export below for training-seed and selection lineage.

## Read An Existing Study Export

Optionally set `MARL_BGS_STUDY_DATA` to a directory produced by
`training.export_study_data`. The reader verifies its index and table hashes.
Keep run lineage and actual training-seed fields when comparing methods;
continuation children and retries are not independent training seeds.
This cell neither creates a study nor re-analyzes raw runs. See
[comparing methods across seeds](https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/docs/training.md#compare-methods-across-seeds) for units and grouping.

The report contains only derived previews. It does not change source results.
For new rich game outputs, use the
[saved-model evaluation](https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/docs/evaluation.md#evaluate-a-saved-model) separately.

In [ ]:
study_path = os.environ.get("MARL_BGS_STUDY_DATA")
if study_path:
    from marl_battlegrounds import training

    study_files = training.load_study_data(study_path)
    report["study_files"] = {name: str(path) for name, path in study_files.items()}
    for name in ("run_index.csv", "selection/recipes.csv"):
        if name in study_files:
            print(name)
            display(pd.read_csv(study_files[name], nrows=5, keep_default_na=False))
else:
    print(
        "No study export supplied. These game results are not training-seed evidence."
    )
(report_dir / "analysis_preview.json").write_text(
    json.dumps(clean(report), indent=2, allow_nan=False) + "\n"
)
print("Saved report files:")
for path in sorted(report_dir.iterdir()):
    if path.is_file():
        print(path)
print("Original results remain at:", result_dir)